# Step 3 -- Pre-Causal Checks

Verify that the selected features actually satisfy the assumptions required by LiNGAM / PC / GES.

| Check | Why it matters |
|---|---|
| Stationarity (ADF) | LiNGAM assumes a stable distribution |
| Non-Gaussianity (kurtosis) | LiNGAM's ICA step relies on non-Gaussian noise |
| Autocorrelation (ACF) | Strong serial dependence -> consider differencing or VARLINGAM |
| Pair distributions | Visual sanity check -- no structural anomalies |
| Bucketing | Split into time windows for robustness checking |

**Input**: `step2_selected.csv`  
**Output**: `step3_bucket_*.csv` (one per time bucket)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
import warnings
warnings.filterwarnings('ignore')

# -- CHANGE THESE --------------------------------------------------------------
FILE_PATH     = r""                      # output CSV from notebook 02
TS_COL        = "date_time"             # timestamp column or None
N_BUCKETS     = 3                       # number of time buckets (early / mid / late)
BUCKET_ROWS   = None  # rows per bucket -- overrides N_BUCKETS if set  (e.g. 10_000)
OUTPUT_PREFIX = "step3_bucket"          # output files: step3_bucket_0.csv, etc.
# ------------------------------------------------------------------------------

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_style('whitegrid')
print('Config OK')

## 1 -- Load

In [ ]:
df = pd.read_csv(FILE_PATH)

if TS_COL and TS_COL in df.columns:
    df[TS_COL] = pd.to_datetime(df[TS_COL], errors='coerce')
    df = df.sort_values(TS_COL).reset_index(drop=True)

features = [c for c in df.columns
            if c != TS_COL and pd.api.types.is_numeric_dtype(df[c])]

print(f'Rows     : {len(df):,}')
print(f'Features : {features}')
df.head()

## 2 -- Stationarity (Augmented Dickey-Fuller Test)

**H_0**: the series has a unit root (non-stationary).  
**p < 0.05** -> reject H_0 -> series is stationary [OK]  
**p >= 0.05** -> non-stationary -- consider differencing before feeding to LiNGAM.

ADF uses a `maxlag` chosen by AIC internally.

In [ ]:
adf_results = []
for col in features:
    series = df[col].dropna()
    try:
        stat, pval, used_lag, nobs, crit, _ = adfuller(series, autolag='AIC')
        adf_results.append({'Feature': col, 'ADF stat': round(stat, 4),
                            'p-value': round(pval, 5), 'Lag': used_lag,
                            'Stationary': '[OK]' if pval < 0.05 else '[X]  NON-STATIONARY'})
    except Exception as e:
        adf_results.append({'Feature': col, 'ADF stat': None, 'p-value': None,
                            'Lag': None, 'Stationary': f'Error: {e}'})

adf_df = pd.DataFrame(adf_results)
print(adf_df.to_string(index=False))

non_stat = [r['Feature'] for r in adf_results if '[X]' in str(r['Stationary'])]
if non_stat:
    print(f'\n[!]  Non-stationary features: {non_stat}')
    print('   Consider first-differencing these columns before causal discovery.')
else:
    print('\n[OK]  All features appear stationary.')

## 3 -- Non-Gaussianity (Kurtosis)

LiNGAM works by exploiting non-Gaussian noise.  
- **Excess kurtosis ~= 0** -> Gaussian (LiNGAM will still run, but identifiability weakens)  
- **|excess kurtosis| >> 0** -> non-Gaussian -> strong signal for LiNGAM [OK]

Bar chart: taller = more non-Gaussian.

In [ ]:
kurt = {col: stats.kurtosis(df[col].dropna(), fisher=True) for col in features}  # excess kurtosis
kurt_s = pd.Series(kurt).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(max(8, len(features)), 4))
colors = ['steelblue' if abs(v) > 0.5 else 'lightgrey' for v in kurt_s.values]
ax.bar(range(len(kurt_s)), kurt_s.values, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline( 0.5, color='red', linestyle='--', linewidth=1, alpha=0.6, label='|kurtosis| = 0.5')
ax.axhline(-0.5, color='red', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xticks(range(len(kurt_s)))
ax.set_xticklabels(kurt_s.index, rotation=90, fontsize=8)
ax.set_ylabel('Excess kurtosis')
ax.set_title('Non-Gaussianity  (grey bars ~= Gaussian, may weaken LiNGAM identifiability)')
ax.legend()
plt.tight_layout()
plt.show()

print(kurt_s.round(4).to_string())

## 4 -- QQ Plots

Visual check for normality: if points lie on the diagonal, the distribution is Gaussian.  
S-curves, heavy tails, or sharp elbows indicate non-Gaussianity -- good for LiNGAM.

In [ ]:
nf  = len(features)
ncg = min(4, nf)
nrg = (nf + ncg - 1) // ncg

fig, axs = plt.subplots(nrg, ncg, figsize=(5 * ncg, 4 * nrg))
axs = np.array(axs).flatten()

for i, col in enumerate(features):
    data = df[col].dropna()
    (osm, osr), (slope, intercept, _) = stats.probplot(data, dist='norm')
    axs[i].scatter(osm, osr, s=6, alpha=0.5, color='steelblue')
    lo, hi = min(osm), max(osm)
    axs[i].plot([lo, hi], [slope * lo + intercept, slope * hi + intercept],
                color='red', linewidth=1.2, linestyle='--')
    k = kurt[col]
    axs[i].set_title(f'{col[:24]}  (k={k:.2f})', fontsize=8)
    axs[i].set_xlabel('Theoretical', fontsize=7)
    axs[i].set_ylabel('Observed',   fontsize=7)

for j in range(nf, len(axs)):
    axs[j].set_visible(False)

fig.suptitle('QQ Plots  (red line = Gaussian; deviations = non-Gaussianity)', fontsize=11)
plt.tight_layout()
plt.show()

## 5 -- Autocorrelation (ACF)

LiNGAM treats each observation as i.i.d. Strong autocorrelation violates this.  
- Bars within the shaded cone -> no significant autocorrelation at that lag [OK]  
- Bars consistently outside the cone -> correlated over time -> use **VARLiNGAM** or difference the series.

In [ ]:
nf  = len(features)
ncg = min(3, nf)
nrg = (nf + ncg - 1) // ncg

fig, axs = plt.subplots(nrg, ncg, figsize=(6 * ncg, 3.5 * nrg))
axs = np.array(axs).flatten()

for i, col in enumerate(features):
    series = df[col].dropna()
    plot_acf(series, ax=axs[i], lags=min(40, len(series) // 4 - 1), title=col[:30], alpha=0.05)
    axs[i].set_xlabel('Lag')

for j in range(nf, len(axs)):
    axs[j].set_visible(False)

fig.suptitle('ACF  (bars outside cone = significant serial correlation)', fontsize=11)
plt.tight_layout()
plt.show()

## 6 -- Pairwise Scatter (coloured by time)

Time-coloured scatter matrix of all selected features.  
Blue = early observations, orange = late. Drifting clusters indicate non-stationarity.

In [ ]:
plot_df = df[features].dropna().reset_index(drop=True)
n_pts   = len(plot_df)
color   = plt.cm.plasma(np.linspace(0.1, 0.9, n_pts))

nf = len(features)
fig, axs = plt.subplots(nf, nf, figsize=(2.8 * nf, 2.8 * nf))

for i, ci in enumerate(features):
    for j, cj in enumerate(features):
        ax = axs[i][j]
        if i == j:
            ax.hist(plot_df[ci], bins=30, color='steelblue', alpha=0.7)
            ax.set_title(ci[:20], fontsize=7)
        else:
            ax.scatter(plot_df[cj], plot_df[ci], c=color, s=3, alpha=0.5, rasterized=True)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle('Pairwise scatter (blue=early -> orange=late)', fontsize=10, y=1.002)
plt.tight_layout()
plt.show()

## 7 -- Assumption Summary Table

Consolidated go/no-go for each feature.

In [ ]:
rows = []
for col in features:
    adf_row  = next((r for r in adf_results if r['Feature'] == col), {})
    p        = adf_row.get('p-value', None)
    k        = kurt.get(col, 0)

    stat_ok  = (p < 0.05) if p is not None else None
    kurt_ok  = abs(k) > 0.5

    flag = ''
    if stat_ok is False: flag += '[non-stationary] '
    if not kurt_ok:      flag += '[near-Gaussian] '
    if not flag:         flag = 'OK'

    rows.append({'Feature': col,
                 'ADF p': round(p, 4) if p is not None else '?',
                 'Stationary': '[OK]' if stat_ok else '[X]',
                 'Excess kurtosis': round(k, 3),
                 'Non-Gaussian': '[OK]' if kurt_ok else '~',
                 'Notes': flag})

summary_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 60)
print(summary_df.to_string(index=False))

## 8 -- Time Bucketing

Split the data into `N_BUCKETS` consecutive windows (or `BUCKET_ROWS`-row chunks).  
Running causal discovery on each bucket separately and comparing the resulting graphs is the standard robustness check:
- **Stable DAG** across buckets -> causal structure is consistent over time [OK]  
- **Changing DAG** -> the data is non-stationary or the causal graph itself evolves

Each bucket is saved as a separate CSV.

In [ ]:
n = len(df)
if BUCKET_ROWS:
    # fixed-size chunks
    edges = list(range(0, n, BUCKET_ROWS)) + [n]
else:
    edges = np.linspace(0, n, N_BUCKETS + 1, dtype=int).tolist()

buckets = []
for k, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
    chunk = df.iloc[lo:hi].copy().reset_index(drop=True)
    buckets.append(chunk)
    path  = f'{OUTPUT_PREFIX}_{k}.csv'
    chunk.to_csv(path, index=False)
    t_lo  = chunk[TS_COL].min() if (TS_COL and TS_COL in chunk.columns) else lo
    t_hi  = chunk[TS_COL].max() if (TS_COL and TS_COL in chunk.columns) else hi - 1
    print(f'  Bucket {k}: rows {lo:,}-{hi:,}  ({t_lo} -> {t_hi})  -> {path}')

print(f'\n{len(buckets)} buckets saved.')

In [ ]:
# Stacked overview: colour each bucket segment differently so you can see the split points
bucket_colors = plt.cm.Set2.colors
col_to_plot   = features[0]

fig, ax = plt.subplots(figsize=(15, 4))
for k, chunk in enumerate(buckets):
    x = chunk[TS_COL] if (TS_COL and TS_COL in chunk.columns) else range(len(chunk))
    ax.plot(x, chunk[col_to_plot],
            linewidth=0.9, color=bucket_colors[k % len(bucket_colors)],
            label=f'Bucket {k}')

ax.set_title(f'{col_to_plot} -- coloured by bucket')
ax.legend(fontsize=9)
if TS_COL and TS_COL in df.columns:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 9 -- Next Steps

The bucket CSVs are now ready to feed into a causal discovery algorithm.

**Recommended path:**

```python
import lingam

df_causal = pd.read_csv('step3_bucket_0.csv').drop(columns=['date_time'], errors='ignore')

model = lingam.DirectLiNGAM()      # or lingam.VARLiNGAM() for time-series
model.fit(df_causal.values)

print('Causal order:', model.causal_order_)
print('Adjacency:\n', model.adjacency_matrix_.round(3))
```

Run on each bucket and compare the adjacency matrices -- stable edges are the reliable causal claims.